# 01. Cleaning and exploratory analysis: Online Retail II

## 1. Business understanding
The company is a UK online retailer of giftware, and many of its customers are themselves trade buyers. Leadership wants to know where revenue comes from, how much of it can be trusted, and which customers and products need action. Clustering comes later, in notebook 02, once these questions have answers.

## 2. Dataset loading and documentation
The source is `online_retail_II.xlsx`, with two sheets covering 1 Dec 2009 to 9 Dec 2011. Each row is **a line on an invoice, not an order**. The fields are Invoice, StockCode, Description, Quantity, InvoiceDate, Price, Customer ID and Country. Prices are in pounds sterling.

Reading the Excel file takes over a minute, so `src/build_data.py` does it once and writes parquet tables. This notebook reads those tables, and the logic lives in `src/cleaning.py` and `src/analytics.py`.

In [1]:
import json, sys
from pathlib import Path
import numpy as np, pandas as pd
import plotly.express as px, plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "notebook_connected"
pio.templates.default = "plotly_white"
pd.set_option("display.max_columns", 40, "display.width", 180, "display.float_format", lambda v: f"{v:,.2f}")
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
OUT = ROOT / "data" / "processed"
sales = pd.read_parquet(OUT / "sales.parquet"); returns = pd.read_parquet(OUT / "returns.parquet")
rfm = pd.read_parquet(OUT / "rfm.parquet"); products = pd.read_parquet(OUT / "products.parquet")
invoices = pd.read_parquet(OUT / "invoices.parquet"); cohorts = pd.read_parquet(OUT / "cohorts.parquet")
gaps = pd.read_parquet(OUT / "gaps.parquet"); affinity = pd.read_parquet(OUT / "affinity.parquet")
audit = json.loads((OUT / "audit.json").read_text()); evals = json.loads((OUT / "cluster_eval.json").read_text())
sales["CustomerID"] = sales["CustomerID"].astype(str)
print(f"{len(sales):,} clean sale lines, {sales['Invoice'].nunique():,} invoices, {len(rfm):,} scored customers")

1,003,340 clean sale lines, 39,516 invoices, 5,839 scored customers


## 3. Data audit
What the raw data looks like, before any decision.

In [2]:
raw = pd.read_parquet(OUT / "raw.parquet")
print(raw.shape); print(raw.isna().sum())
print("Exact duplicate rows:", raw.duplicated().sum())
print("Invoices starting with C:", raw["Invoice"].astype(str).str.startswith("C").sum())
print("Invoices starting with A:", raw["Invoice"].astype(str).str.startswith("A").sum())
print("Price <= 0:", (raw["Price"] <= 0).sum(), "  Quantity < 0:", (raw["Quantity"] < 0).sum())
raw[["Quantity", "Price"]].describe(percentiles=[.01, .5, .99])

(1044848, 8)
Invoice             0
StockCode           0
Description      4275
Quantity            0
InvoiceDate         0
Price               0
CustomerID     235287
Country             0
dtype: int64


Exact duplicate rows: 11812
Invoices starting with C: 19165


Invoices starting with A: 6
Price <= 0: 6029   Quantity < 0: 22557


,Quantity,Price
count,"1,044,848.00","1,044,848.00"
mean,9.99,4.59
std,174.22,121.70
min,"-80,995.00","-53,594.36"
1%,-3.00,0.21
50%,3.00,2.10
99%,100.00,18.00
max,"80,995.00","38,970.00"


**Reading this.** The raw sheets overlap: both contain 1 to 9 Dec 2010, so simply joining them double counts a week. Extreme quantities and negative values also need explaining before they are removed.

## 4. Cleaning and data quality decisions

| Issue | Decision | Business reason |
|---|---|---|
| Sheets overlap on 1 to 9 Dec 2010 | Keep the second sheet for that week | Otherwise a peak-season week counts twice |
| Exact duplicate rows | Removed, with their value recorded | Most likely double scans. If any are real, revenue is slightly understated |
| Missing Customer ID | Grouped by invoice first, then kept for revenue and left out of RFM | They are whole guest checkouts and still real sales |
| `C` invoices (cancellations) | Kept in a separate table and matched to earlier sales | Removing them would overstate revenue |
| Price of 0 or less | Isolated | Giveaways, damage write-offs and stock adjustments are not sales |
| `A` invoices (bad debt) | Isolated | Accounting entries |
| Postage, fees and manual entries | Isolated | Postage should not rank as a top product |

In [3]:
missing = raw["CustomerID"].isna()
g = missing.groupby(raw["Invoice"].astype(str)).agg(["sum", "size"])
print("Invoices with no ID on any line:", ((g["sum"] == g["size"])).sum())
print("Invoices mixing ID and no ID:   ", (((g["sum"] > 0) & (g["sum"] < g["size"]))).sum())
pd.Series(audit, name="value").to_frame()

Invoices with no ID on any line: 8752
Invoices mixing ID and no ID:    0


,value
raw_rows_sheet1,525461
raw_rows_sheet2,541910
overlap_rows_removed,22523
rows_after_overlap,1044848
exact_duplicate_rows,11812
exact_duplicate_value,"57,481.47"
guest_rows,235151
guest_invoices_all_lines,8752
guest_invoices_mixed,0
total_invoices,53628


**Guest checkouts.** No invoice mixes lines with and without an ID, so missing IDs are a property of the whole invoice and not random gaps. That supports keeping them for revenue and products. They cannot enter RFM, because there is no customer to score.

**Cancellations.** Each return is matched to the same customer's most recent earlier purchase of the same item, and the quantity is taken off that sale. A return with no earlier purchase in the data, because the sale predates Dec 2009 or the buyer has no ID, stays unmatched: it counts in the return rate but is not subtracted from net revenue.

In [4]:
print(f"Return lines fully matched: {audit['return_rows_fully_matched']:,} of {audit['return_rows_goods']:,}")
print(f"Gross revenue {audit['gross_revenue']:,.0f}, net revenue {audit['net_revenue']:,.0f}")
print(f"Unmatched return value {audit['return_value_unmatched']:,.0f}")
returns.sort_values("ReturnValue", ascending=False).head(5)[["Invoice", "Description", "ReturnQty", "Price", "CustomerID", "MatchedQty"]]

Return lines fully matched: 15,876 of 17,914
Gross revenue 19,643,862, net revenue 18,913,459
Unmatched return value 40,603


,Invoice,Description,ReturnQty,Price,CustomerID,MatchedQty
1043360,C581484,"PAPER CRAFT , LITTLE BIRDIE",80995,2.08,16446,80995
564562,C541433,MEDIUM CERAMIC TOP STORAGE JAR,74215,1.04,12346,74215
663083,C550456,FAIRY CAKE FLANNEL ASSORTED COLOUR,3114,2.10,15749,3114
663081,C550456,WHITE HANGING HEART T-LIGHT HOLDER,1930,2.55,15749,1930
663079,C550456,DOORMAT FAIRY CAKE,670,6.75,15749,670


## 5. Feature engineering
`TotalLineRevenue = Quantity * Price`, `NetRevenue` after matched returns, date parts (`Year`, `Month`, `DayOfWeek`, `Hour`), and the `Is_UK` flag. CustomerID is stored as text, so it can never enter a model as a number.

In [5]:
sales[["Invoice", "StockCode", "Quantity", "Price", "TotalLineRevenue", "ReturnedQty", "NetRevenue", "Year", "Month", "DayOfWeek", "Hour", "Is_UK"]].head()

,Invoice,StockCode,Quantity,Price,TotalLineRevenue,ReturnedQty,NetRevenue,Year,Month,DayOfWeek,Hour,Is_UK
0,489434,85048,12,6.95,83.40,0,83.40,2009,12,1,7,True
1,489434,79323P,12,6.75,81.00,0,81.00,2009,12,1,7,True
2,489434,79323W,12,6.75,81.00,0,81.00,2009,12,1,7,True
3,489434,22041,48,2.10,100.80,0,100.80,2009,12,1,7,True
4,489434,21232,24,1.25,30.00,0,30.00,2009,12,1,7,True


## 6. EDA
### Sales over time: how does revenue change, and is there a season?

In [6]:
m = sales.groupby("YearMonth")["NetRevenue"].sum().reset_index()
px.bar(m, x="YearMonth", y="NetRevenue", title="Monthly net revenue (December 2011 stops on the 9th)").show()
y1 = sales[(sales.InvoiceDate >= "2009-12-01") & (sales.InvoiceDate < "2010-12-01")].NetRevenue.sum()
y2 = sales[(sales.InvoiceDate >= "2010-12-01") & (sales.InvoiceDate < "2011-12-01")].NetRevenue.sum()
cal = sales[sales.Year == 2010]
print(f"Like-for-like years: {y1:,.0f} then {y2:,.0f} ({y2 / y1 - 1:+.1%})")
print(f"Oct to Dec share of 2010: {cal[cal.Month >= 10].NetRevenue.sum() / cal.NetRevenue.sum():.1%}")

Like-for-like years: 9,151,773 then 9,316,839 (+1.8%)
Oct to Dec share of 2010: 35.2%


**Reading this.** Revenue climbs from September and peaks in November both years. Orders peak before December, which fits retailers stocking up ahead of Christmas. The last bar is a partial month and must not be read as a fall.

### Time: which months, weekdays and hours are strongest?

In [7]:
d = pd.DataFrame({"Day": invoices.InvoiceDate.dt.day_name(), "Hour": invoices.InvoiceDate.dt.hour})
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
grid = d.groupby(["Day", "Hour"]).size().unstack(fill_value=0).reindex(order).fillna(0)
px.imshow(grid, aspect="auto", title="Orders by weekday and hour", color_continuous_scale="Blues").show()
print(d.Day.value_counts())
sat = invoices[invoices.InvoiceDate.dt.day_name() == "Saturday"]
print("Saturday orders fall on", sat.InvoiceDate.dt.date.nunique(), "date:", sat.InvoiceDate.dt.date.unique())

Day
Thursday     8097
Tuesday      7091
Wednesday    7072
Monday       6197
Friday       5938
Sunday       4801
Saturday       30
Name: count, dtype: int64
Saturday orders fall on 1 date: [datetime.date(2009, 12, 5)]


**Reading this.** Thursday is the strongest day, and around noon is the peak. Saturday has only 30 orders in two years, all on one date, so this looks like a store policy, not low demand. Nights and Saturdays suit maintenance. Weekday mornings are the natural slot to test ads, though these data show when people order and not whether ads would work then.

### Products: units, revenue, and is 80/20 true here?

In [8]:
p = products[products.Revenue > 0].sort_values("Revenue", ascending=False)
display(p.head(10)[["StockCode", "Description", "Revenue", "Units", "Orders"]])
display(products.sort_values("Units", ascending=False).head(10)[["StockCode", "Description", "Units", "Revenue", "AvgPrice"]])
cum = p.Revenue.cumsum() / p.Revenue.sum(); x = np.arange(1, len(p) + 1) / len(p)
n80 = (cum <= .8).sum() + 1
px.line(x=x, y=cum, labels={"x": "Share of products", "y": "Cumulative revenue"}, title=f"Pareto curve: {n80 / len(p):.1%} of products make 80% of revenue").show()

,StockCode,Description,Revenue,Units,Orders
1632,22423,REGENCY CAKESTAND 3 TIER,"314,163.22",25036,3918
4152,85123A,WHITE HANGING HEART T-LIGHT HOLDER,"251,877.43",91113,5464
4130,85099B,JUMBO BAG RED RETROSPOT,"180,595.64",96015,4013
3127,47566,PARTY BUNTING,"147,121.03",27925,2674
3890,84879,ASSORTED COLOUR BIRD ORNAMENT,"128,621.39",79612,2807
1320,22086,PAPER CHAIN KIT 50'S CHRISTMAS,"116,556.49",34616,2018
3391,79321,CHILLI LIGHTS,"80,003.58",15719,1135
1427,22197,SMALL POPCORN HOLDER,"78,922.16",87698,2322
1601,22386,JUMBO BAG PINK POLKADOT,"75,459.59",38956,2245
252,20725,LUNCH BAG RED RETROSPOT,"69,764.29",39409,3056


,StockCode,Description,Units,Revenue,AvgPrice
3480,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,104651,"24,113.93",0.29
4130,85099B,JUMBO BAG RED RETROSPOT,96015,"180,595.64",1.95
609,21212,PACK OF 72 RETROSPOT CAKE CASES,93942,"51,326.03",0.55
4152,85123A,WHITE HANGING HEART T-LIGHT HOLDER,91113,"251,877.43",2.95
1427,22197,SMALL POPCORN HOLDER,87698,"78,922.16",0.85
3890,84879,ASSORTED COLOUR BIRD ORNAMENT,79612,"128,621.39",1.69
110,17003,BROCADE RING PURSE,69630,"14,646.11",0.29
1240,21977,PACK OF 60 PINK PAISLEY CAKE CASES,55843,"27,961.83",0.55
4009,84991,60 TEATIME FAIRY CAKE CASES,53702,"26,862.04",0.55
17,15036,ASSORTED COLOURS SILK FAN,43355,"32,332.00",0.83


**Reading this.** The classic 80/20 rule is a close description, not an exact one: about 22% of products make 80% of revenue. The units list and the revenue list differ, because cheap items lead on volume. Plan stock from units and margin from revenue.

### Dead stock: high earlier volume, nothing in the final 90 days

In [9]:
dead = products[products.IsDead].sort_values("UnitsBefore90", ascending=False)
print(len(dead), "products; earlier units:", f"{dead.UnitsBefore90.sum():,.0f}")
dead.head(10)[["StockCode", "Description", "UnitsBefore90", "UnitsLast90", "LastSold"]]

139 products; earlier units: 557,657


,StockCode,Description,UnitsBefore90,UnitsLast90,LastSold
2969,37410,BLACK AND WHITE PAISLEY FLOWER MUG,"25,701.00",0.00,2010-04-07 14:10:00
3518,84270,ASSTD DESIGN BUBBLE GUM RING,"21,286.00",0.00,2010-12-06 16:57:00
39,16033,MINI HIGHLIGHTER PENS,"14,881.00",0.00,2011-02-17 15:40:00
507,21085,SET/6 WOODLAND PAPER CUPS,"13,731.00",0.00,2010-07-12 12:44:00
513,21091,SET/6 WOODLAND PAPER PLATES,"13,700.00",0.00,2010-03-18 12:47:00
520,21099,SET/6 STRAWBERRY PAPER CUPS,"13,194.00",0.00,2010-05-07 12:13:00
514,21092,SET/6 STRAWBERRY PAPER PLATES,"12,846.00",0.00,2010-03-17 13:09:00
444,20993,JAZZ HEARTS MEMO PAD,"10,967.00",0.00,2010-03-09 12:48:00
1653,22444,GROW YOUR OWN PLANT IN A CAN,"10,021.00",0.00,2011-05-19 10:47:00
4275,85220,SMALL FAIRY CAKE FRIDGE MAGNETS,"9,732.00",0.00,2010-02-17 10:51:00


**Reading this.** The final 90 days cover the seasonal peak, so a quiet season does not explain this. Many stopped selling long before, so some are probably deliberate range changes. The data has no stock levels, so this list says what to check, not what is actually sitting in the warehouse.

### Customers: who contributes the most?

In [10]:
display(rfm.sort_values("Monetary", ascending=False).head(10)[["CustomerID", "Country", "Monetary", "Frequency"]])
display(rfm.sort_values("Frequency", ascending=False).head(10)[["CustomerID", "Country", "Frequency", "Monetary"]])
s = rfm.Monetary.sort_values(ascending=False)
print(f"Top 1% of customers: {s.iloc[:len(s) // 100].sum() / s.sum():.1%} of registered revenue")
print(f"Top 10% of customers: {s.iloc[:len(s) // 10].sum() / s.sum():.1%}")
rfm[["Recency", "Frequency", "Monetary", "AOV", "Products"]].describe(percentiles=[.25, .5, .75, .9])

,CustomerID,Country,Monetary,Frequency
5654,18102,United Kingdom,"578,408.64",145
2253,14646,Netherlands,"522,885.29",144
1771,14156,EIRE,"297,875.18",142
2511,14911,EIRE,"258,630.86",370
5014,17450,United Kingdom,"233,643.91",49
1315,13694,United Kingdom,"190,130.56",141
5073,17511,United Kingdom,"168,728.06",59
67,12415,Australia,"143,107.02",23
4261,16684,United Kingdom,"141,524.24",54
2658,15061,United Kingdom,"124,972.29",126


,CustomerID,Country,Frequency,Monetary
2511,14911,EIRE,370,"258,630.86"
390,12748,United Kingdom,315,"46,759.52"
5396,17841,United Kingdom,211,"67,304.34"
2907,15311,United Kingdom,205,"111,778.47"
725,13089,United Kingdom,200,"109,932.25"
2213,14606,United Kingdom,183,"29,056.27"
5404,17850,United Kingdom,155,"50,659.85"
5654,18102,United Kingdom,145,"578,408.64"
2253,14646,Netherlands,144,"522,885.29"
1771,14156,EIRE,142,"297,875.18"


Top 1% of customers: 30.9% of registered revenue
Top 10% of customers: 63.1%


,Recency,Frequency,Monetary,AOV,Products
count,"5,839.00","5,839.00","5,839.00","5,839.00","5,839.00"
mean,199.91,6.22,"2,798.11",358.12,81.85
std,208.61,12.63,"13,821.81",468.91,116.03
min,0.00,1.00,2.90,2.90,1.00
25%,25.00,1.00,334.81,173.86,19.00
50%,95.00,3.00,844.86,273.35,45.00
75%,379.00,7.00,"2,180.02",405.27,103.00
90%,530.60,13.00,"5,250.90",614.69,197.00
max,738.00,370.00,"578,408.64","12,425.45","2,505.00"


**Typical customer.** The median customer has about 3 orders and spends roughly £845, last ordering about three months before the data ends. The mean spend is far higher than the median because a few accounts are very large, so the median is the better description of a typical customer.

### Geography: which countries lead?

In [11]:
g = sales.groupby("Country", observed=True).agg(Revenue=("NetRevenue", "sum"), Orders=("Invoice", "nunique"))
g["Customers"] = rfm.groupby("Country").size()
g = g.sort_values("Revenue", ascending=False); g["Share"] = g.Revenue / g.Revenue.sum()
display(g.head(12))
intl = g.drop("United Kingdom").reset_index()
px.choropleth(intl, locations="Country", locationmode="country names", color="Revenue", title="Revenue outside the UK (EIRE and RSA need the mapping used in the app)", color_continuous_scale="Blues").show()

,Revenue,Orders,Customers,Share
Country,,,,
United Kingdom,"16,150,654.02",36184,"5,326.00",0.85
EIRE,"604,573.83",581,3.00,0.03
Netherlands,"545,777.98",216,22.00,0.03
Germany,"375,024.33",753,106.00,0.02
France,"294,466.82",598,93.00,0.02
Australia,"166,528.50",89,14.00,0.01
Switzerland,"92,922.56",85,22.00,0.00
Spain,"84,833.16",144,35.00,0.00
Sweden,"84,359.87",99,19.00,0.00


C:\Users\ammmb\AppData\Local\Temp\ipykernel_23028\1668850029.py:6: DeprecationWarning:

The library used by the *country names* `locationmode` option is changing in an upcoming version. Country names in existing plots may not work in the new version. To ensure consistent behavior, consider setting `locationmode` to *ISO-3*.



**Reading this.** The UK makes about 85% of revenue, so overall trends are mostly UK trends. EIRE's high revenue from very few customers suggests a handful of trade accounts, which is a concentration risk worth checking.

### Baskets: is there wholesale behaviour?

In [12]:
print(invoices[["Value", "Units", "Lines"]].describe(percentiles=[.5, .9, .99]))
bulk = invoices[invoices.IsBulk]
print(f"Bulk cut-off: {invoices.Units.quantile(.9):,.0f} units. Bulk is {len(bulk) / len(invoices):.0%} of orders and {bulk.Value.sum() / invoices.Value.sum():.0%} of revenue")
px.histogram(invoices[invoices.Value < 5000], x="Value", nbins=80, title="Order value (orders under £5,000)").show()

          Value     Units     Lines
count 39,226.00 39,226.00 39,226.00
mean     482.17    273.74     25.14
std    1,091.48    967.83     42.36
min        0.19      1.00      1.00
50%      295.38    147.00     15.00
90%      893.68    520.00     51.00
99%    4,008.61  2,172.75    197.75
max   52,940.94 87,167.00  1,106.00
Bulk cut-off: 520 units. Bulk is 10% of orders and 42% of revenue


**Reading this.** A median order of nearly 300 units is far beyond household shopping, so much of this business is trade buying. The top tenth of orders by units bring about 42% of revenue.

### Returns: how common, and who is involved?

In [13]:
print(f"Returned value is {audit['return_value_total'] / audit['gross_revenue']:.1%} of gross sales")
display(products[products.SoldUnits >= 500].sort_values("ReturnRate", ascending=False).head(10)[["Description", "SoldUnits", "ReturnedUnits", "ReturnRate"]])
r = returns[returns.CustomerID.astype(str) != ""].groupby("CustomerID", observed=True).ReturnValue.sum().sort_values(ascending=False)
display(r.head(10).to_frame())
print(f"Top 10 customers make {r.head(10).sum() / r.sum():.0%} of customer-linked return value")

Returned value is 3.6% of gross sales


,Description,SoldUnits,ReturnedUnits,ReturnRate
2792,"PAPER CRAFT , LITTLE BIRDIE",80995,"80,995.00",1.00
2347,MEDIUM CERAMIC TOP STORAGE JAR,78033,"74,494.00",0.95
3223,ROSES ON BLUE TEACUP CANDLE,539,504.00,0.94
2294,PANTRY CHOPPING BOARD,1154,946.00,0.82
2243,IVORY CHANDELIER T-LIGHT HOLDER,657,432.00,0.66
3405,WILLOW BRANCH LIGHTS.,725,448.00,0.62
2244,FLOWERS CHANDELIER T-LIGHT HOLDER,963,579.00,0.60
2245,BEADED CHANDELIER T-LIGHT HOLDER,773,433.00,0.56
3579,BLUE/PINK NEST STRIPE BOX,1142,567.00,0.50
3926,PSYCHEDELIC METAL SIGN CALENDAR,975,480.00,0.49


,ReturnValue
CustomerID,
16446,"168,469.60"
12346,"77,183.60"
15749,"22,998.40"
12931,"20,800.37"
14911,"13,730.36"
14277,"12,829.34"
16029,"12,554.16"
12454,"12,360.82"
17450,"11,140.34"


Top 10 customers make 51% of customer-linked return value


## 7. Business insights
These are computed from the data above. They describe what happened and do not establish why.

1. **Seasonality.** Q4 brings well over a third of annual revenue, peaking in November. Commit stock and carrier capacity before September.
2. **Growth.** Comparing like-for-like 12-month windows gives a truer picture than comparing calendar years, because December 2011 is incomplete.
3. **Product concentration.** About 22% of products make 80% of revenue. Protect availability there first.
4. **Long tail.** The last 5% of revenue comes from about half the range, which makes it the first place to review for range cuts.
5. **Geography.** The UK is about 85% of revenue. EIRE, the Netherlands and Germany lead abroad.
6. **Wholesale behaviour.** A tenth of orders bring about 42% of revenue, so volume pricing fits better than consumer promotions.
7. **Customer concentration.** The top 10% of customers bring about 63% of registered revenue, and losing one large account would show in the monthly figures.
8. **Ordering rhythm.** Nearly all orders arrive on weekdays and Sundays in working hours, which gives clear maintenance windows.
9. **Returns.** Returns are a modest 3.6% of gross but concentrated, and single large cancellations cause the spikes.
10. **Dead stock.** 139 high-volume lines sold nothing in the final 90 days, during the peak. Check which were delisted on purpose.
11. **Guest orders.** About 13% of sales have no customer record, so they cannot be targeted later.
12. **Repeat buying.** The median gap between orders is about 25 days, a better trigger for reminders than a fixed monthly send.

Continue in `02_rfm_clustering.ipynb`.